In [1]:
import os
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.16.1


In [2]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

dataset_path = "../../datasets/food-101/images"

print(dataset_path)
print(os.path.exists(dataset_path))

../../datasets/food-101/images
True


In [3]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("Datasets Loaded")

Found 101000 files belonging to 101 classes.
Using 80800 files for training.
Found 101000 files belonging to 101 classes.
Using 20200 files for validation.
Datasets Loaded


In [4]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)

print("Dataset Ready")

Dataset Ready


In [5]:
model = load_model("../saved_models/efficientnetb0_advanced.keras")

print("Advanced Fine-tuned model loaded successfully")

Advanced Fine-tuned model loaded successfully


In [6]:
base_model = model.layers[2]

print(base_model.name)
print("Total Layers:", len(base_model.layers))

efficientnetb0
Total Layers: 238


In [7]:
base_model.trainable = True

for layer in base_model.layers[:-100]:
    layer.trainable = False

print(
    "Trainable Layers:",
    sum(layer.trainable for layer in base_model.layers)
)

Trainable Layers: 60


In [8]:
base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = True

print("All Trainable:", sum(layer.trainable for layer in base_model.layers))

All Trainable: 238


In [9]:
for layer in base_model.layers[:-100]:
    layer.trainable = False

print(
    "Trainable Layers:",
    sum(layer.trainable for layer in base_model.layers)
)

Trainable Layers: 100


In [10]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=2e-6
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("100 Layers Fine-Tune Compiled")

100 Layers Fine-Tune Compiled


In [11]:
checkpoint_phase4 = ModelCheckpoint(
    "../saved_models/efficientnetb0_100layers.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

In [12]:
early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

csv_logger = CSVLogger(
    "../saved_models/training_log_100layers.csv"
)

print("Callbacks Ready")

Callbacks Ready


In [13]:
history_phase4 = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20,
    callbacks=[
        checkpoint_phase4,
        early_stop,
        reduce_lr,
        csv_logger
    ],
    verbose=1
)

Epoch 1/20
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 856ms/step - accuracy: 0.6862 - loss: 1.1648
Epoch 1: val_accuracy improved from None to 0.74475, saving model to ../saved_models/efficientnetb0_100layers.keras

Epoch 1: finished saving model to ../saved_models/efficientnetb0_100layers.keras
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 4867s 957ms/step - accuracy: 0.6888 - loss: 1.1527 - val_accuracy: 0.7448 - val_loss: 0.9774 - learning_rate: 2.0000e-06
Epoch 2/20
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 959ms/step - accuracy: 0.6950 - loss: 1.1377
Epoch 2: val_accuracy improved from 0.74475 to 0.74673, saving model to ../saved_models/efficientnetb0_100layers.keras

Epoch 2: finished saving model to ../saved_models/efficientnetb0_100layers.keras
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 5328s 1s/step - accuracy: 0.6966 - loss: 1.1268 - val_accuracy: 0.7467 - val_loss: 0.9681 - learning_rate: 2.0000e-06
Epoch 3/20
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 738ms/step - accuracy: 0.6970 - loss: 1.1323
Epoch 3: val_accuracy improved